# 6. Fluctuation statistics (N=500^2)

We compute it from scratch (for that we first need the average data, obtained from roughness calculations

In [1]:
import numpy as np
import pandas as pd
from TracyWidom import TracyWidom
import os

from surface_functions_2D import compute_surface_variance, compute_surface_roughness
from fluctuation_functions_2D import compute_rescaled_phases_at_index, normalized_tw_pdf

In [16]:
z_dict = {"TimeDep": {"EW": 2, "KPZ": 3/2}, "Columnar": {"EW": 2, "KPZ": 1.36}}
alpha_dict = {"TimeDep": {"EW": 1/2, "KPZ": 1/2}, "Columnar": {"EW": 3/2, "KPZ": 1.07}}
alpha_s_dict = {"TimeDep": {"EW": 1/2, "KPZ": 1/2}, "Columnar": {"EW": 3/2, "KPZ": 1.4}}  # anomalous scaling, only for columnar noise
alpha_loc_dict = {"TimeDep": {"EW": 1/2, "KPZ": 1/2}, "Columnar": {"EW": 1, "KPZ": 0.96}} # anomalous scaling, only for columnar noise
beta_dict = {"TimeDep": {"EW": alpha_dict["TimeDep"]["EW"]/z_dict["TimeDep"]["EW"], "KPZ": alpha_dict["TimeDep"]["KPZ"]/z_dict["TimeDep"]["KPZ"]}, 
             "Columnar": {"EW": alpha_dict["Columnar"]["EW"]/z_dict["Columnar"]["EW"], "KPZ": alpha_dict["Columnar"]["KPZ"]/z_dict["Columnar"]["KPZ"]}}
delta_to_label = {"0": "EW", "atan5": "KPZ"}

In [17]:
# Adjustment to the crossover time: t_x = tx * L**z (BY HAND for Ks=[1,40])
tx = {"TimeDep": {"EW": None, "KPZ": None}, "Columnar": {"EW": 1/600, "KPZ": 1/6}}

In [18]:
beta_dict = {"Columnar": {"EW": 0.5, "KPZ": 0.783}}

## Tracy-Widom preloading

In [4]:
## Loading TW pre-saved data
from scipy.interpolate import interp1d

data = np.loadtxt('../tw_data.csv', delimiter=',', skiprows=1)
s_vals = data[:, 0]
ds = s_vals[1] - s_vals[0]
tw_interp = {}

# Create interpolators for each TW distribution
for col, beta in enumerate([1, 2, 4], start=1):
    pdf = data[:, col]
    # Calculate moments from CSV to standardize
    mu = np.sum(s_vals * pdf) * ds
    sigma = np.sqrt(np.sum(s_vals**2 * pdf) * ds - mu**2)
    # Map to standardized coordinates
    tw_interp[beta] = interp1d((s_vals - mu) / sigma, pdf * sigma, 
                               kind='cubic', bounds_error=False, fill_value=0.0)


def normalized_tw_pdf(x, beta, grid_min=-20.0, grid_max=10.0, n_grid=200_000):
    """
    Standardized Tracy–Widom PDF (mean 0, variance 1) for beta=1,2,4.

    Returns f_std(x) where if Y ~ TW_beta, then X = (Y - mu)/sigma has PDF:
        f_std(x) = sigma * f_Y(sigma*x + mu)
    """
    TW = TracyWidom(beta=beta)

    # Build pdf function
    pdf_fn = lambda y: TW.pdf(y)

    mu, sigma = _estimate_moments_from_pdf(
        pdf_fn, grid_min=grid_min, grid_max=grid_max, n=n_grid
    )
    return TW.pdf(sigma * x + mu) * sigma

## Dataframe computations

In [19]:
summary_rows = []
for typenoise in ["TimeDep", "Columnar"]:

    if typenoise=="TimeDep":
        continue
    
    for delta, deltaname in zip([0, np.arctan(5)], ["0", "atan5"]):

        path = f"../Simulations_2D/Output{typenoise}/all_simulations_delta{deltaname}.pkl"
        if not os.path.exists(path):
            print(f"File {path} not found, skipping.")
            continue


        # Load previously saved simulations
        df = pd.read_pickle(path)

        roughness_list = []  # used to compute per-sim roughness
        variance_list = []   # used to compute per-sim variance
        datapoint_list = []

        skip = 0
        # Loop over simulations (one per seed / row)
        for _, row in df.iterrows():
            t = row["t"]
            phases = row["phases"]
            
            # if row["L"] != 500:
            #     skip += 1
            #     continue

            # ensure array, not list-of-lists
            variance = np.asarray(compute_surface_variance(phases), dtype=float)
            roughness = np.asarray(compute_surface_roughness(phases), dtype=float)

            variance_list.append(variance)
            roughness_list.append(roughness)
            datapoint_list.append(len(roughness))
        
        print(f"## {typenoise} – Delta {deltaname} -- Skipped: {skip}")

        # Avoid chained-assignment issues
        df = df.copy()
        df["roughness"] = roughness_list
        df["datapoints"] = datapoint_list

        # DO NO SAVEEE
        # Save back the updated simulations DataFrame (with roughness column)
        # df.to_pickle(path)

        ## AVERAGE ROUGHNESS PER (K, L, T) ##
        # group by K, L, T and compute mean roughness for each group
        for (K_val, L_val, T_val), group in df.groupby(["K", "L", "T"]):

            if L_val != 500:
                continue

            rough_arrays = [np.asarray(r, dtype=float) for r in group["roughness"]]
            mean_roughness = np.mean(np.stack(rough_arrays, axis=0), axis=0)
            # assume all t are identical within group; take the first
            t = group.iloc[0]["t"]
            seed_list = group.index.tolist()
            M = len(seed_list) 

            avg_id = f"{typenoise}_delta{deltaname}_L{L_val}_K{K_val}_T{T_val}_M{M}"

            summary_rows.append(
                {
                    "avg_id": avg_id,
                    "typenoise": typenoise,
                    "delta": delta,
                    "deltaname": deltaname,
                    "K": K_val,
                    "L": L_val,
                    "t": t,
                    "M": M,
                    "T": T_val,
                    "mean_roughness": mean_roughness,
                }
            )

## Columnar – Delta 0 -- Skipped: 0
## Columnar – Delta atan5 -- Skipped: 0


In [20]:
# Note: we start from scratch the average DataFrame (each time this code is run)
avg_df = pd.DataFrame()

# Build/update AverageSims dataframe
if summary_rows:
    new_avg = pd.DataFrame(summary_rows).set_index("avg_id")
    if avg_df.empty:
        avg_df = new_avg
    else:
        # ensure we have an index; if not, give it one
        if avg_df.index.name is None:
            avg_df = avg_df.copy()
            avg_df.index.name = "avg_id"
        avg_df = pd.concat([avg_df, new_avg])
        # keep last occurrence per avg_id
        avg_df = avg_df[~avg_df.index.duplicated(keep="last")]

    # DO NOT SAVE!!

In [21]:
len(avg_df)

2

In [22]:
summary_rows = []
for typenoise in ["TimeDep", "Columnar"]:

    if typenoise=="TimeDep":
        continue
    
    for delta, deltaname in zip([0, np.arctan(5)], ["0", "atan5"]):

        path = f"../Simulations_2D/Output{typenoise}/all_simulations_delta{deltaname}.pkl"
        if not os.path.exists(path):
            print(f"File {path} not found, skipping.")
            continue

        print(f"## {typenoise} – Delta {deltaname}")

        # Load previously saved simulations
        df = pd.read_pickle(path)

        beta = beta_dict[typenoise][delta_to_label[deltaname]]
        z = z_dict[typenoise][delta_to_label[deltaname]]

        ## AVERAGE FLUCTUATION STATISTICS PER (K, L, T) ##
        # group by K, L, T and compute fluctuation statistics for each group
        for (K_val, L_val, T_val), group in df.groupby(["K", "L", "T"]):

            if L_val != 500:
                continue
                
            t_cross = tx[typenoise][delta_to_label[deltaname]] * (L_val ** z)
            
            per_seed_varphis = []
            for _, row in group.iterrows():
                t = np.asarray(row["t"])
                phases = np.asarray(row["phases"])

                # t0_idx = int(np.argmin(np.abs(t - 50)))-2 # optional, where to start.
                t0_idx = 0 # optional, where to start.

                for ti_idx in range(t0_idx + 1, len(t)):
                    if t[ti_idx] < t_cross:
                        v = compute_rescaled_phases_at_index(phases, t, t0_idx, ti_idx, beta=beta)
                        per_seed_varphis.append(v)

            varphis_full = np.ravel(per_seed_varphis)
            avg_id = f"{typenoise}_delta{deltaname}_L{L_val}_K{K_val}_T{T_val}_M{len(group)}"

            summary_rows.append(
                {
                    "avg_id": avg_id,
                    "varphis_full": varphis_full
                }
            )

## Columnar – Delta 0
## Columnar – Delta atan5


In [23]:
avg_df_path = "../Simulations_2D/fluctuations_N500_columnar.pkl"
# avg_df = pd.read_pickle(avg_df_path)

new_avg = pd.DataFrame(summary_rows).set_index("avg_id")
avg_df.loc[new_avg.index, "varphis_full"] = new_avg["varphis_full"]

avg_df.to_pickle(avg_df_path)

In [24]:
avg_df

,typenoise,delta,deltaname,K,L,t,M,T,mean_roughness,varphis_full
avg_id,,,,,,,,,,
Columnar_delta0_L500_K20.0_T20000_M31,Columnar,0.000000,0,20.0,500,"[0.0, 1.00000050000025, 1.6000008000004002, 2....",31,20000,"[0.0, 0.04363913486127933, 0.05484272893896487...","[0.01716723691586557, 0.0015467259716610501, -..."
Columnar_deltaatan5_L500_K20.0_T20000_M19,Columnar,1.373401,atan5,20.0,500,"[0.0, 1.00000050000025, 1.6000008000004002, 2....",19,20000,"[0.0, 0.10205084143959577, 0.1283351397168975,...","[0.09987178956399922, 0.0763349856181436, 0.04..."
